# Evaluating Regression Models Using MSE and R²

Mean Squared Error and R² answer two different questions in regression: how large are the errors, and how much variance does the model explain?

This lesson explains both metrics, how to compare them against a baseline, and how to interpret them together without confusing absolute error with explanatory power.

## Why MSE and R² matter

MSE shows the size of squared prediction errors. R² shows how much better the model is than predicting the mean for everyone.

Use them together because:

- MSE tells you how large the errors are.
- R² tells you how much target variation the model explains.
- A model can look good on one metric and weak on the other.
- Baseline comparison makes both metrics meaningful.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score, train_test_split

rng = np.random.default_rng(67)
n_rows = 500

df = pd.DataFrame({
    'size_sqft': rng.integers(500, 3500, size=n_rows),
    'bedrooms': rng.integers(1, 6, size=n_rows),
    'age_years': rng.integers(1, 40, size=n_rows),
})

df['price'] = (
    df['size_sqft'] * 1150
    + df['bedrooms'] * 14000
    - df['age_years'] * 1600
    + rng.normal(0, 28000, size=n_rows)
).round(2)

df.head()

## Baseline comparison

Always compare the model against the mean baseline on the same held-out test set. The baseline gives R² its meaning and makes MSE easier to interpret.

In [ ]:
TARGET_COLUMN = 'price'
X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

baseline = DummyRegressor(strategy='mean')
baseline.fit(X_train, y_train)
baseline_preds = baseline.predict(X_test)

model = LinearRegression()
model.fit(X_train, y_train)
model_preds = model.predict(X_test)

def evaluate(name, y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    print(f'{name:25s} | MSE: {mse:8.2f} | RMSE: {rmse:6.2f} | R2: {r2:.3f}')

evaluate('Baseline (mean)', y_test, baseline_preds)
evaluate('Linear Regression', y_test, model_preds)

## Cross-validation

Cross-validation gives a more stable view of performance than one split. R² uses the standard scoring convention directly, while MSE must be negated because scikit-learn expects higher scores to be better.

In [ ]:
cv_r2 = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')
cv_mse = -cross_val_score(model, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
cv_rmse = np.sqrt(cv_mse)

print(f'CV R2 scores:   {np.round(cv_r2, 3)}')
print(f'Mean CV R2:     {cv_r2.mean():.3f} +/- {cv_r2.std():.3f}')
print(f'CV RMSE scores: {np.round(cv_rmse, 3)}')
print(f'Mean CV RMSE:   {cv_rmse.mean():.3f} +/- {cv_rmse.std():.3f}')

baseline_mse = mean_squared_error(y_test, baseline_preds)
model_mse = mean_squared_error(y_test, model_preds)
print({
    'baseline_mse': round(baseline_mse, 3),
    'model_mse': round(model_mse, 3),
    'baseline_r2': round(r2_score(y_test, baseline_preds), 3),
    'model_r2': round(r2_score(y_test, model_preds), 3),
})

## Practical checklist

- Compute metrics on held-out data only.
- Compare both MSE and R² against a baseline.
- Interpret MSE in target units squared, and prefer RMSE when communicating results.
- Treat negative R² as a warning sign.
- Use cross-validation to check stability across folds.

## Bonus resources

- Scikit-learn Model Evaluation Guide
- Regression Metrics - Scikit-learn